# ColdSite-DTI — does mixed precision change the answer? (Colab, T4)

**The question.** KIBA is to be trained with mixed precision (`--amp`): it measured 2.0x
faster for HyperAttentionDTI on a T4 (`results/speed_test_kiba_t4.md`), which is the
difference between KIBA fitting in the quota or not. DAVIS was trained in full precision.
Before KIBA's numbers are put beside DAVIS's, we check that `--amp` does not by itself move
the two things the paper reports: **test AUROC** and **precision@10** against binding sites.

**The check.** Train HyperAttentionDTI on DAVIS `cold_pair`, 3 seeds, with `--amp`, and
put the result beside the same 3 cells trained in full precision by the DAVIS 36-grid on
Kaggle. Same split, same seeds, same batch, same early stopping; only the arithmetic
differs.

| why this cell | |
|---|---|
| HyperAttentionDTI | the largest speed-up, so the most arithmetic done in half precision |
| `cold_pair` | the smallest validation set (264 rows) and the noisiest level: if half precision nudges which epoch is kept, it shows here first |

**How to read it.** Three seeds cannot prove two things equal. What they can show is
whether `--amp` moves the mean by more than full precision already moves between seeds.
Within that spread: use `--amp` for KIBA and say so in Methods. Outside it: stop and
discuss before KIBA starts.

## Before you run

**Runtime -> Change runtime type -> T4 GPU.** Then run the cells in order. Section 2 asks
you to authorise Google Drive -- that click is yours.

Expect **~1-1.5 hours per seed** (about 25-40 epochs at ~2 min each), so 3-4.5 hours.

## Safe to interrupt

This notebook trains on branch `kiba-resume`, whose trainers save their state to Drive after
every epoch. If Colab disconnects, run the notebook again from the top: finished seeds are
skipped and an interrupted one **continues from its last finished epoch**. This is also
the first real test of that resume on a GPU.

## Why the outputs are renamed

The trainer names an `--amp` cell exactly like the full-precision grid cell. Mixed into the
grid's folder it would silently replace a real one. So every finished output is renamed
with an `_ampcheck` suffix and kept in its own Drive folder, `coldsite-amp-validation`
(`colab_analysis.ipynb` refuses `_ampcheck` files). **Never copy these into the grid's
results.**

## 1. Check the runtime

In [ ]:
import time, torch

assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
p = torch.cuda.get_device_properties(0)
print(f'GPU   : {p.name}, {p.total_memory/1e9:.1f} GB')
print(f'torch : {torch.__version__}')

# The DAVIS 36-grid's HyperAttentionDTI setting, so only the precision differs:
# batch 32 on a >= 14 GB card (a T4 is 15 GB), else 8 x 4 accumulation.
big = p.total_memory / 1e9 >= 14
HAT_BATCH, HAT_ACCUM = (32, 1) if big else (8, 4)
print(f'batch : {HAT_BATCH} x accum {HAT_ACCUM}')
if not big:
    print('NOTE: the grid cells ran at 32 x 1 on a 15 GB T4. 8 x 4 is the same effective batch '
          'but not the same arithmetic -- prefer a T4 so only the precision differs.')

## 2. Mount Drive

`WORK` holds the training state while a seed runs, so a disconnect loses at most one epoch.
Finished, renamed outputs sit one level up.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS = '/content/drive/MyDrive/coldsite-amp-validation'
WORK = f'{RESULTS}/work'
os.makedirs(WORK, exist_ok=True)
print('results ->', RESULTS)
print('finished:', sorted(f for f in os.listdir(RESULTS) if f.endswith('_results.json')))
print('in progress:', sorted(os.listdir(WORK)) or 'nothing')

## 3. Clone the branch

`kiba-resume` carries the `--amp` flag and epoch-level resume. The cell refuses a checkout
without them.

In [ ]:
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
BRANCH = 'kiba-resume'
SRC = '/content/ColdSite-DTI_New'
if not os.path.exists(SRC):
    !git clone --branch {BRANCH} {REPO} {SRC}
os.chdir(SRC)
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip install -q tabulate subword-nmt

assert os.path.exists('src/model/resume.py') and os.path.exists('src/model/precision.py'), (
    f'{BRANCH} has no resume/precision modules -- is it pushed? Re-run this cell.')
!git log --oneline -1

## 4. Data and splits

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
!python -m src.data.build_splits 2>&1 | grep -E 'davis|leakage'

import pandas as pd
got = tuple(len(pd.read_csv(f'data/splits/davis/cold_pair/{part}.csv'))
            for part in ('train', 'valid', 'test'))
assert got == (15190, 264, 1144), f'cold_pair split differs from the rest of the project: {got}'
print(f'davis/cold_pair {got} OK -- the same rows the grid trained on.')

## 5. Train the three seeds with `--amp`, and score their explanations

Per seed: train (or continue) in `WORK`; score precision@10 on the cold-pair test proteins
with the same `run_ladder` the paper uses (one test pair per protein); then rename
everything with `_ampcheck` and move it out of `WORK`.

In [ ]:
import glob, json, shutil, subprocess, sys

SEEDS = [1, 2, 3]
SPLIT = 'cold_pair'
MODEL = 'hyperattentiondti'
SUFFIX = '_ampcheck'
GROUND_TRUTH = 'data/davis_ground_truth_sites.json'


def stream(cmd):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
    return proc.wait()


def renamed(name, seed):
    return name.replace(f'seed{seed}_{MODEL}', f'seed{seed}_{MODEL}{SUFFIX}') \
               .replace(f'davis_seed{seed}', f'davis_seed{seed}{SUFFIX}')


def ladder(checkpoint_dir, seed, out_dir):
    """precision@10 for one seed's cold-pair checkpoint; returns the ladder json path."""
    code_ = stream([sys.executable, '-u', '-m', 'src.evaluation.run_ladder',
                    '--model', MODEL, '--dataset', 'davis', '--task', 'binary',
                    '--seed', str(seed), '--checkpoint-dir', checkpoint_dir,
                    '--ground-truth', GROUND_TRUTH, '--out-dir', out_dir, '--device', 'cuda'])
    found = glob.glob(f'{out_dir}/ladder_*davis_seed{seed}.json')
    assert code_ == 0 and len(found) == 1, f'ladder failed for seed {seed}'
    return found[0]


def final_results(seed):
    return f'{RESULTS}/davis_{SPLIT}_binary_seed{seed}_{MODEL}{SUFFIX}_results.json'


for seed in SEEDS:
    if os.path.exists(final_results(seed)) and glob.glob(f'{RESULTS}/ladder_*davis_seed{seed}{SUFFIX}.json'):
        print(f'seed {seed}: already done')
        continue
    print(f'\n{"=" * 70}\nHyperAttentionDTI, davis/{SPLIT}, seed {seed}, --amp\n{"=" * 70}')
    code_ = stream([sys.executable, '-u', '-m', 'src.model.train_hyperattentiondti',
                    '--split-dir', f'data/splits/davis/{SPLIT}', '--dataset', 'davis',
                    '--split', SPLIT, '--seed', str(seed),
                    '--batch-size', str(HAT_BATCH), '--accum-steps', str(HAT_ACCUM),
                    '--patience', '15', '--min-epochs', '10', '--epochs', '100', '--amp',
                    '--checkpoint-dir', WORK, '--results-dir', WORK, '--skip-if-done'])
    if code_ != 0:
        print(f'!! seed {seed} exited {code_} -- re-run this cell to continue it')
        continue
    payload = json.load(open(f'{WORK}/davis_{SPLIT}_binary_seed{seed}_{MODEL}_results.json'))
    assert payload.get('amp') is True, 'this cell was not trained with --amp'

    ladder(WORK, seed, WORK)
    for path in glob.glob(f'{WORK}/*'):
        name = os.path.basename(path)
        if f'seed{seed}' in name and not name.endswith('_resume.pt'):
            shutil.move(path, f'{RESULTS}/{renamed(name, seed)}')
    print(f'seed {seed}: test AUROC {payload["test_metrics"]["auroc"]:.4f}, '
          f'best epoch {payload["best_epoch"]} -> {RESULTS}')

## 6. Compare with the full-precision grid cells

Put account 1's three HyperAttentionDTI cold-pair cells from the DAVIS 36-grid in a Drive
folder -- for each seed the `.pt` and its `_results.json`:

    coldsite_dti_davis_cold_pair_binary_seed{1,2,3}_hyperattentiondti.pt
    davis_cold_pair_binary_seed{1,2,3}_hyperattentiondti_results.json

and set `FP32_DIR` to it. Their files are only read, never changed; their ladder is scored
here with the same code as the `--amp` cells.

In [ ]:
import statistics as st

FP32_DIR = None      # e.g. '/content/drive/MyDrive/coldsite-grid36-kaggle'


def p_at_10(ladder_path):
    return json.load(open(ladder_path))[SPLIT]['by_k']['10']['precision_at_k']


amp = {}
for seed in SEEDS:
    ladders = glob.glob(f'{RESULTS}/ladder_*davis_seed{seed}{SUFFIX}.json')
    if os.path.exists(final_results(seed)) and ladders:
        payload = json.load(open(final_results(seed)))
        amp[seed] = (payload['test_metrics']['auroc'], p_at_10(ladders[0]), payload['best_epoch'])

fp32 = {}
if FP32_DIR:
    for seed in SEEDS:
        ckpt = f'{FP32_DIR}/coldsite_dti_davis_{SPLIT}_binary_seed{seed}_{MODEL}.pt'
        res = f'{FP32_DIR}/davis_{SPLIT}_binary_seed{seed}_{MODEL}_results.json'
        if not (os.path.exists(ckpt) and os.path.exists(res)):
            print(f'fp32 seed {seed}: not in FP32_DIR')
            continue
        payload = json.load(open(res))
        assert not payload.get('amp'), f'{res} says it was trained with --amp'
        # the ladder reads only this one checkpoint, from a scratch copy
        scratch = f'/content/fp32_seed{seed}'
        shutil.rmtree(scratch, ignore_errors=True)
        os.makedirs(scratch)
        shutil.copy2(ckpt, scratch)
        fp32[seed] = (payload['test_metrics']['auroc'], p_at_10(ladder(scratch, seed, scratch)),
                      payload['best_epoch'])

print(f"\n{'seed':>4}  {'AUROC fp32':>10}  {'AUROC amp':>9}  {'p@10 fp32':>9}  {'p@10 amp':>8}  "
      f"{'best ep fp32/amp':>16}")
for seed in SEEDS:
    f, a = fp32.get(seed), amp.get(seed)
    cell = lambda v, i, w: f'{v[i]:{w}.4f}' if v else f'{"--":>{w}}'
    print(f"{seed:>4}  {cell(f, 0, 10)}  {cell(a, 0, 9)}  {cell(f, 1, 9)}  {cell(a, 1, 8)}  "
          f"{(str(f[2]) if f else '--'):>8}/{(str(a[2]) if a else '--'):<7}")

if len(amp) == 3 and len(fp32) == 3:
    print()
    verdicts = []
    for i, metric in enumerate(('test AUROC', 'precision@10')):
        f_vals, a_vals = [fp32[s][i] for s in SEEDS], [amp[s][i] for s in SEEDS]
        delta = st.mean(a_vals) - st.mean(f_vals)
        spread = st.stdev(f_vals)
        within = abs(delta) <= spread
        verdicts.append(within)
        print(f'{metric:13s} fp32 {st.mean(f_vals):.4f} +- {spread:.4f}   '
              f'amp {st.mean(a_vals):.4f} +- {st.stdev(a_vals):.4f}   '
              f'difference {delta:+.4f}  ->  '
              + ('within the full-precision seed spread' if within else 'OUTSIDE the full-precision seed spread'))
    print('\n' + ('Mixed precision does not move either number beyond seed-to-seed noise: use --amp '
                   'for KIBA, and state it in Methods.' if all(verdicts) else
                   'Mixed precision moves at least one number beyond seed-to-seed noise: do NOT start '
                   'KIBA with --amp yet -- send these numbers to discuss first.'))
    print('Send this table either way.')
else:
    print(f'\n{len(amp)}/3 --amp seeds done, {len(fp32)}/3 full-precision seeds found '
          f'-- the verdict needs all six.')